
# Skadi BI - Silver

Fase inicial: lê Bronze, corrige tipos, padroniza os valores e interrompe a execução se houver dados inválidos.

In [ ]:
from pyspark.sql import DataFrame, functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

PERFIS_VALIDOS = ["OPERADOR", "GESTOR", "ADMIN", "SUPER_ADMIN"]
CATEGORIAS_VALIDAS = ["DUVIDA", "SUGESTAO", "RECLAMACAO", "OUTRO"]
STATUS_CHAMADO_VALIDOS = ["REGISTRADO", "EM_ATENDIMENTO", "ATENDIDO"]


def validar_sem_registros(df: DataFrame, regra: str) -> None:
    if df.limit(1).count() > 0:
        raise ValueError(f"Validacao Silver falhou: {regra}")


def gravar_silver(df: DataFrame, nome_tabela: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

In [ ]:
usuarios_silver = (
    spark.table("bronze_usuario")
    .dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id"),
        F.upper(F.trim("nivel_acesso")).alias("nivel_acesso"),
    )
)

chamados_silver = (
    spark.table("bronze_solicitacao_suporte")
    .dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id"),
        F.col("cod_usuario_solicitante").cast("long").alias("cod_usuario_solicitante"),
        F.col("cod_usuario_responsavel").cast("long").alias("cod_usuario_responsavel"),
        F.upper(F.trim("categoria")).alias("categoria"),
        F.upper(F.trim("status")).alias("status"),
        F.col("data_hora_abertura").cast("timestamp").alias("data_hora_abertura"),
        F.col("data_hora_encerramento").cast("timestamp").alias("data_hora_encerramento"),
    )
)

rpa_silver = (
    spark.table("bronze_controle_rpa")
    .dropDuplicates(["nome_carga"])
    .select(
        F.trim("nome_carga").alias("nome_carga"),
        F.col("ultima_execucao_sucesso").cast("timestamp").alias("ultima_execucao_sucesso"),
    )
)

In [ ]:
validar_sem_registros(
    usuarios_silver.where(~F.col("nivel_acesso").isin(PERFIS_VALIDOS)),
    "nivel de acesso invalido",
)
validar_sem_registros(
    usuarios_silver.where(F.col("id").isNull() | F.col("nivel_acesso").isNull()),
    "usuario sem identificacao ou nivel de acesso",
)
validar_sem_registros(
    chamados_silver.where(~F.col("categoria").isin(CATEGORIAS_VALIDAS)),
    "categoria invalida",
)
validar_sem_registros(
    chamados_silver.where(
        F.col("id").isNull()
        | F.col("cod_usuario_solicitante").isNull()
        | F.col("categoria").isNull()
        | F.col("status").isNull()
    ),
    "chamado com campo obrigatorio ausente",
)
validar_sem_registros(
    chamados_silver.where(~F.col("status").isin(STATUS_CHAMADO_VALIDOS)),
    "status invalido",
)
validar_sem_registros(
    chamados_silver.where(F.col("data_hora_abertura").isNull()),
    "abertura ausente",
)
validar_sem_registros(
    chamados_silver.where(
        F.col("data_hora_encerramento").isNotNull()
        & (F.col("data_hora_encerramento") < F.col("data_hora_abertura"))
    ),
    "encerramento anterior a abertura",
)
validar_sem_registros(
    chamados_silver.where(
        (F.col("status") == "ATENDIDO")
        & F.col("data_hora_encerramento").isNull()
    ),
    "chamado atendido sem encerramento",
)

gravar_silver(usuarios_silver, "silver_usuario")
gravar_silver(chamados_silver, "silver_solicitacao_suporte")
gravar_silver(rpa_silver, "silver_controle_rpa")

In [ ]:
display(spark.table("silver_usuario"))
display(spark.table("silver_solicitacao_suporte"))
display(spark.table("silver_controle_rpa"))